# Run MetaGeneFormer on the three-species hippocampus dataset

## Make the run csv file

In [ ]:
# Make the dataset-specific SATURN run CSV
from pathlib import Path
import pandas as pd

df = pd.DataFrame({
    "path": ['D:/111icde_addition_experiments/3_species_hippocampus/processed_h5ad/human.h5ad', 'D:/111icde_addition_experiments/3_species_hippocampus/processed_h5ad/pig.h5ad', 'D:/111icde_addition_experiments/3_species_hippocampus/processed_h5ad/macaM.h5ad'],
    "species": ['human', 'pig', 'macaM'],
    "embedding_path": ['D:/protein_go_ontology_embeddings/human_protein_go_fused.fused_embedding.pkl.gz', 'D:/protein_go_ontology_embeddings/pig_protein_go_fused.fused_embedding.pkl.gz', 'D:/protein_go_ontology_embeddings/macaca_mulatta_protein_go_fused.fused_embedding.pkl.gz'],
})

for column in ["path", "embedding_path"]:
    missing = [path for path in df[column] if not Path(path).exists()]
    if missing:
        raise FileNotFoundError(f"Missing {column} files: {missing}")

df.to_csv("data/hippocampus_run.csv", index=False)
df

In [ ]:
!cd ../../ ; python3 saturn_multiple_seeds.py \
                --run=Vignettes/3_species_hippocampus/data/hippocampus_run.csv \
                --gpus 1 3 4 \
                --seeds=1 # change to number of seeds you want to run

In [ ]:
# Use the latest SATURN pretrain output for final cross-species integration
from pathlib import Path

saturn_result_dir = Path("../multiple_seeds_results/saturn_results")
pretrain_files = list(saturn_result_dir.glob("*hippocampus_run*_seed_0_pretrain.h5ad"))
if not pretrain_files:
    raise FileNotFoundError(
        f"No hippocampus seed-0 pretrain h5ad found in {saturn_result_dir.resolve()}"
    )

pretrain_h5ad = max(pretrain_files, key=lambda path: path.stat().st_mtime).resolve()
metageneformer_outdir = Path("metageneformer_results").resolve()
metageneformer_outdir.mkdir(parents=True, exist_ok=True)

print("MetaGeneFormer input:", pretrain_h5ad)
print("MetaGeneFormer output:", metageneformer_outdir)

!cd ../../ ; python3 metageneformer.py \
                --input_h5ad="{pretrain_h5ad}" \
                --outdir="{metageneformer_outdir}" \
                --species_key=species \
                --label_key=labels2 \
                --device=cuda:0 \
                --epochs=50 \
                --seed=2025 \
                --output_prefix=hippocampus_metageneformer

In [ ]:
# Visualize the latest MetaGeneFormer-integrated output
from pathlib import Path
import scanpy as sc

integrated_files = list(Path("metageneformer_results").glob(
    "hippocampus_metageneformer_fraction_*.h5ad"
))
if not integrated_files:
    raise FileNotFoundError("Run the MetaGeneFormer cell before visualization")

integrated_h5ad = max(integrated_files, key=lambda path: path.stat().st_mtime)
integrated_ad = sc.read_h5ad(integrated_h5ad)
sc.pl.umap(integrated_ad, color="labels2")
sc.pl.umap(integrated_ad, color="species")